In [2]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1. Data: 5D inputs and 1D outputs (Function 6)
# =========================================================

X_raw = np.array([
    [0.7281861 , 0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.24238435, 0.84409997, 0.5778091 , 0.67902128, 0.50195289],
    [0.72952261, 0.7481062 , 0.67977464, 0.35655228, 0.67105368],
    [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
    [0.6188123 , 0.33180214, 0.18728787, 0.75623847, 0.3288348 ],
    [0.78495809, 0.91068235, 0.7081201 , 0.95922543, 0.0049115 ],
    [0.14511079, 0.8966846 , 0.89632223, 0.72627154, 0.23627199],
    [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
    [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
    [0.75759436, 0.35583141, 0.0165229 , 0.4342072 , 0.11243304],
    [0.5367969 , 0.30878091, 0.41187929, 0.38822518, 0.5225283 ],
    [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
    [0.6293079 , 0.80348368, 0.81140844, 0.04561319, 0.11062446],
    [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
    [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
    [0.25890557, 0.79367771, 0.6421139 , 0.19667346, 0.59310318],
    [0.43216593, 0.71561781, 0.3418191 , 0.70499988, 0.61496184],
    [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
    [0.9217762 , 0.93187122, 0.41487637, 0.59505727, 0.73562569],
    [0.12667892, 0.2914703 , 0.06452848, 0.6805146 , 0.89281919],
    [1.057739  , 1.031871  , 1.078805  , 1.061655  , 0.992819  ],
    [0.183405  , 0.304243  , 0.524756  , 0.431945  , 0.29123   ],
    [0.268807  , 0.268756  , 0.495982  , 0.986904  , 0.010463  ],
    [0.071886  , 0.119564  , 0.11427   , 0.97486   , 0.062381  ],
])

y_raw = np.array([
    6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
    2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
    8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
    6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
    2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
    7.71337361e+03, 1.61662575e+03, 5.63309324e+02, 3.46190426e+03
])

# =========================================================
# 2. Device & Seeds
# =========================================================

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 3. Scaling
# =========================================================

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = x_scaler.fit_transform(X_raw)
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32, device=device)

# =========================================================
# 4. Surrogate Network (Deeper + Dropout)
# =========================================================

class SurrogateNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=128, dropout_p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SurrogateNN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=2e-3, weight_decay=2e-4)

# =========================================================
# 5. Training with Early Stopping
# =========================================================

def train_model(model, X, y, n_epochs=2500, patience=400):
    n_samples = X.shape[0]
    n_train = max(int(0.8 * n_samples), 1)

    indices = torch.randperm(n_samples)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    best_state = None
    best_val = float("inf")
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = criterion(val_pred, y_val).item()

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model

model = train_model(model, X_tensor, y_tensor)

# =========================================================
# 6. MC Dropout Prediction
# =========================================================

def predict_mc(model, X, n_samples=100):
    model.train()  # keep dropout active
    X_scaled = x_scaler.transform(X)
    X_t = torch.tensor(X_scaled, dtype=torch.float32, device=device)

    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(X_t).cpu().numpy())

    preds = np.array(preds).squeeze(-1)

    # invert scaling
    preds_flat = preds.reshape(-1, 1)
    preds_orig = y_scaler.inverse_transform(preds_flat).reshape(preds.shape)

    mu = preds_orig.mean(axis=0)
    sigma = preds_orig.std(axis=0)
    return mu, sigma

# =========================================================
# 7. Acquisition (EI & PI) — FIXED NO np.erf
# =========================================================

def normal_pdf(z):
    return np.exp(-0.5 * z**2) / math.sqrt(2 * math.pi)

def normal_cdf(z):
    z = np.asarray(z)
    return 0.5 * (1 + np.vectorize(math.erf)(z / math.sqrt(2)))

def acquisition_ei_pi(mu, sigma, y_best, xi=0.0):
    eps = 1e-9
    sigma = np.maximum(sigma, eps)

    improvement = mu - y_best - xi
    Z = improvement / sigma

    cdf_vals = normal_cdf(Z)
    pdf_vals = normal_pdf(Z)

    ei = improvement * cdf_vals + sigma * pdf_vals
    pi = cdf_vals

    return np.maximum(ei, 0), pi

# =========================================================
# 8. Propose Next Point
# =========================================================

def propose_next_point(model, X_obs, y_obs,
                       n_candidates=20000,
                       xi=0.01,
                       mc_samples=100,
                       random_seed=123):

    rng = np.random.default_rng(random_seed)
    X_cand = rng.random((n_candidates, 5))

    mu, sigma = predict_mc(model, X_cand, n_samples=mc_samples)

    y_min, y_best = float(np.min(y_obs)), float(np.max(y_obs))
    y_range = max(y_best - y_min, 1e-6)
    effective_xi = xi * y_range

    ei, pi = acquisition_ei_pi(mu, sigma, y_best, xi=effective_xi)

    best_idx = int(np.argmax(ei))

    return {
        "x_next": X_cand[best_idx],
        "mu_next": float(mu[best_idx]),
        "sigma_next": float(sigma[best_idx]),
        "pi_next": float(pi[best_idx]),
        "ei_next": float(ei[best_idx]),
        "x_best_obs": X_obs[np.argmax(y_obs)],
        "y_best_obs": float(y_best)
    }

# =========================================================
# 9. Local Sensitivity (Gradients)
# =========================================================

def local_sensitivity(model, x_point):
    model.eval()
    x_scaled = x_scaler.transform(x_point.reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32, device=device, requires_grad=True)

    y_pred = model(x_t)
    y_pred.backward()

    grads = x_t.grad.detach().cpu().numpy().flatten()
    g = np.abs(grads)
    if g.sum() == 0:
        return np.ones_like(g) / len(g)
    return g / g.sum()

# =========================================================
# 10. Main
# =========================================================

def main():
    details = propose_next_point(
        model,
        X_raw,
        y_raw,
        n_candidates=20000,
        xi=0.01,
        mc_samples=100,
        random_seed=RANDOM_SEED
    )

    print("\n=== CURRENT BEST ===")
    print("x* =", details["x_best_obs"])
    print("y* =", details["y_best_obs"])

    print("\n=== NEXT QUERY POINT ===")
    print("x_next =", details["x_next"])
    print("Predicted mean =", details["mu_next"])
    print("Predicted sigma =", details["sigma_next"])
    print("EI =", details["ei_next"])
    print("PI =", details["pi_next"])

    print("\n=== LOCAL SENSITIVITY ===")
    sens = local_sensitivity(model, details["x_next"])
    for i, s in enumerate(sens, 1):
        print(f"Dim {i}: {s:.3f}")

if __name__ == "__main__":
    main()



=== CURRENT BEST ===
x* = [1.057739 1.031871 1.078805 1.061655 0.992819]
y* = 7713.37361

=== NEXT QUERY POINT ===
x_next = [0.98505348 0.91285682 0.96772398 0.99302192 0.94406231]
Predicted mean = 5871.43603515625
Predicted sigma = 777.763427734375
EI = 1.7235248749794732
PI = 0.006804506123976428

=== LOCAL SENSITIVITY ===
Dim 1: 0.083
Dim 2: 0.495
Dim 3: 0.249
Dim 4: 0.169
Dim 5: 0.004
